# getitem-back-add-at — ex2: getitem_back along axis 0 of (N, D) — row scatter-add

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `getitem-back-add-at`. Running the final beacon cell reports progress against the `Backprop: getitem_back via add-at` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: getitem_back via add-at` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`getitem-back-add-at`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "getitem-back-add-at"
DD_SUBTOPIC = "Backprop: getitem_back via add-at"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## 2-D row indexing: `getitem_back` along axis 0 of `(N, D)` — quick refresher

ex1 covered the 1-D case. The deeper facet is the SAME primitive (`index_add_`) on a higher-rank `x`: `out = x[idx]` where `x: (N, D)` and `idx: (K,)` returns a `(K, D)` slice of ROWS.

```python
x.shape   = (N, D)
idx.shape = (K,)         # 1-D long tensor selecting rows
out       = x[idx]       # shape (K, D)
grad_out.shape = (K, D)

grad_in = zeros_like(x)                   # (N, D)
grad_in.index_add_(0, idx, grad_out)      # scatter-add full ROWS
```

Repeated indices still SUM — but now the contribution at each repeated position is a whole length-`D` row, not a scalar. The 1-D case is the `D=1` degenerate version of this.

### Exercise 2 — getitem_back along axis 0 of (N, D) — row scatter-add

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply 2-D row scatter-add: for x: (N, D) and idx: (K,), use index_add_ along axis 0 to accumulate full-row contributions.
> Keywords: getitem, 2d, row-index, index-add, scatter-add
> ```

**KCs targeted:** `getitem-backward-pattern`, `scatter-add-for-repeated-indices`

Implement `getitem_back_rows(grad_out, out, x, idx)` for the forward op `out = x[idx]` where:

- `x`     shape `(N, D)` — embedding table.
- `idx`   shape `(K,)`   — `torch.LongTensor` of row indices.
- `out`   shape `(K, D)` — gathered rows.
- `grad_out` shape `(K, D)`.

Derivation:
- `out[i, :] = x[idx[i], :]`, so `d(out[i, :]) / d(x[j, :]) = I_D` if `j == idx[i]` else `0`.
- Chain rule: `dL/dx[j, :] = sum_{i : idx[i] == j} grad_out[i, :]`.
- Each contribution is a FULL ROW (length D), not a scalar.

Implementation:
1. `grad_in = torch.zeros_like(x)` — shape `(N, D)`.
2. `grad_in.index_add_(0, idx, grad_out)` — scatter-adds the K rows of `grad_out` into the rows of `grad_in` selected by `idx`.
3. Return `grad_in`.

**This is the embedding-layer backward.** Token embeddings, positional embeddings, anywhere you do `x[idx]` to gather rows — the backward is this same scatter-add.

No autograd. Return a `(N, D)` tensor with the same dtype as `x`.

In [ ]:
def getitem_back_rows(grad_out: Tensor, out: Tensor, x: Tensor, idx: Tensor) -> Tensor:
    """dL/dx for out = x[idx], where x is 2-D and idx is 1-D row indices."""
    raise NotImplementedError()


def _test_ex2():
    # --- unique row indices ---
    x = t.tensor([
        [1.0, 2.0, 3.0],
        [4.0, 5.0, 6.0],
        [7.0, 8.0, 9.0],
        [10., 11., 12.],
    ])  # (4, 3)
    idx = t.tensor([0, 2], dtype=t.long)
    out = x[idx]  # (2, 3)
    grad_out = t.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
    g = getitem_back_rows(grad_out, out, x, idx)
    assert g.shape == x.shape, f'shape: {g.shape}'
    expected = t.tensor([
        [1., 2., 3.],
        [0., 0., 0.],
        [4., 5., 6.],
        [0., 0., 0.],
    ])
    assert t.allclose(g, expected), f'unique: {g}'

    # --- repeated row index: contributions SUM as full rows ---
    idx = t.tensor([1, 1, 3], dtype=t.long)
    out = x[idx]
    grad_out = t.tensor([[1., 1., 1.], [2., 2., 2.], [10., 20., 30.]])
    g = getitem_back_rows(grad_out, out, x, idx)
    # row 1 gets [1+2, 1+2, 1+2] = [3, 3, 3]
    # row 3 gets [10, 20, 30]
    expected = t.tensor([
        [0., 0., 0.],
        [3., 3., 3.],
        [0., 0., 0.],
        [10., 20., 30.],
    ])
    assert t.allclose(g, expected), f'repeated rows: {g}'

    # --- all-same row index ---
    idx = t.tensor([2, 2, 2, 2], dtype=t.long)
    out = x[idx]
    grad_out = t.ones(4, 3)
    g = getitem_back_rows(grad_out, out, x, idx)
    # row 2 gets 4 * [1, 1, 1] = [4, 4, 4]
    assert t.allclose(g[2], t.full((3,), 4.0)), f'all-same row 2: {g[2]}'
    assert t.allclose(g[[0, 1, 3]], t.zeros(3, 3)), f'other rows nonzero: {g}'

    # --- conservation: g.sum() == grad_out.sum() ---
    rng = t.Generator().manual_seed(0)
    X = t.randn(8, 5, generator=rng)
    IDX = t.randint(0, 8, (20,), generator=rng, dtype=t.long)
    G = t.randn(20, 5, generator=rng)
    g = getitem_back_rows(G, X[IDX], X, IDX)
    assert g.shape == (8, 5)
    assert abs(g.sum().item() - G.sum().item()) < 1e-4, (
        f'conservation broken: g.sum={g.sum()} grad_out.sum={G.sum()}'
    )

    # --- per-column conservation ---
    for col in range(5):
        assert abs(g[:, col].sum().item() - G[:, col].sum().item()) < 1e-4

    # --- not aliased to grad_out ---
    g_in = t.ones(2, 3)
    g_out = getitem_back_rows(g_in, x[t.tensor([0, 1])], x, t.tensor([0, 1], dtype=t.long))
    assert g_out.data_ptr() != g_in.data_ptr(), 'must not alias grad_out'

    # --- agreement with torch.autograd ---
    x_ref = t.randn(6, 4, requires_grad=True, generator=t.Generator().manual_seed(2))
    idx_ref = t.tensor([0, 2, 0, 5, 2], dtype=t.long)
    y = x_ref[idx_ref].sum()
    y.backward()
    x_det = x_ref.detach()
    g_ours = getitem_back_rows(t.ones(5, 4), x_det[idx_ref], x_det, idx_ref)
    assert t.allclose(g_ours, x_ref.grad, atol=1e-6), (
        f'disagrees with autograd: max diff '
        f'{(g_ours - x_ref.grad).abs().max()}'
    )
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def getitem_back_rows(grad_out: Tensor, out: Tensor, x: Tensor, idx: Tensor) -> Tensor:
    grad_in = t.zeros_like(x)
    # axis 0 scatter — each grad_out row is added to row idx[i] of grad_in.
    grad_in.index_add_(0, idx, grad_out)
    return grad_in
```

**Why `index_add_` not `scatter_add_`.** Both work, but `index_add_` with `dim=0` and 1-D `idx` is the row-scatter primitive — its semantics map 1:1 to the math here. `scatter_add_` needs an `index` tensor of the same shape as `grad_out`, which is overkill.

**Why this is the embedding-layer backward.** A token embedding is exactly `x[idx]` where `x: (vocab_size, embed_dim)`. Every NLP model uses this same back fn to accumulate token-level gradients into the embedding table. Repeated tokens in a sequence get summed.

**Conservation per column.** Each element of `grad_out` lands somewhere in `grad_in`, so column sums match. Useful debugging invariant when shapes get high-rank.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()